In [68]:
from screenshot import get_screen_with_grid
import cv2

In [69]:
images = get_screen_with_grid()

overlayed = images[1]
original  = images[0]

In [78]:
cv2.imshow("overlayed_image", overlayed)
k = cv2.waitKey(0)

# 4. Close all OpenCV windows
cv2.destroyAllWindows()

In [ ]:
from PIL import Image
from google import genai
from dotenv import load_dotenv

load_dotenv()

client = genai.Client()

image = Image.fromarray(overlayed)
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[image, "If you were to be instructed to close the currently open window by the user of the computer shown in the image, how would you go about it? Your response should be strictly a json output. "
    "It should look like this: {'action': 'action goes here' , 'target': 'the cell to be targeted', 'value':'the value to type or the key to click'}. In case of moving the mouse, you would output something like this: "
    "{'action':'click', 'target':'C5'}. In case you also wanted to type, the value would be this: {'action':'click_and_type', 'target':'C5', 'value':'Hello World'}.  "]
)
print(response.text)

```json
{"action": "click", "target": "AL1"}
```


In [71]:
import re

# main_text = "Some intro text. {Desired block of text}. {More text follows.}"
pattern = r"{(.*?)}"

matches = re.findall(pattern, response.text)
instructions = []


for i in matches:
    # Use .strip() to remove spaces and quotes from both the key and the value
    my_dict = {
        key.strip(" '\""): value.strip(" '\"") 
        for item in i.split(',') 
        for key, value in [item.split(':')]
    }
    instructions.append(my_dict)
    # Output: Desired block of text


In [72]:
instructions

[{'action': 'click', 'target': 'AL1'}]

In [83]:
def label_to_coords(label, step_size=50):
    chars = re.findall(r'[A-Za-z]+|\d+', label)

    # print(chars)
    
    letter_part = chars[0].upper()
    number = int(chars[1])

    # Convert standard Excel-style column letters to a number (A=1, Z=26, AA=27)
    letter_index = 0
    for char in letter_part:
        # Multiply current total by 26 and add the character's value (A=1...Z=26)
        letter_index = letter_index * 26 + (ord(char) - 64)

    # print(f"Letter string: {letter_part}, Calculated Index: {letter_index}")

    x = (letter_index * step_size) - (step_size // 2)
    y = (number * step_size) - (step_size // 2)

    return x, y

In [84]:
label_to_coords('AA1')

(1325, 25)

In [86]:
action_to_take = instructions[0]

In [87]:
action_to_take

{'action': 'click', 'target': 'AL1'}

In [90]:
target = label_to_coords(action_to_take['target'])
action = action_to_take['action']
# value = action_to_take['value'] | None



In [ ]:
import pyautogui


if action == 'click':

    pyautogui.moveTo(target)
    pyautogui.click()


elif action == 'click_and_type':
    pyautogui.moveTo(target)
    pyautogui.click()
    pyautogui.typewrite(value)